# 3. EHR to text

Load FHIR R4 patients, anchor their events to encounters, select a cohort, turn
each patient into a text timeline, and send that text through the same
de-identification and chunking as any clinical note.

Needs `pip install "openbtk[ehr]"`.

> Every note, patient, number and identifier in this notebook is **fictitious**. It runs offline, downloads no model, and uses no real patient data.

In [1]:
import os

# Keep OpenBTK's routine debug lines out of this notebook's output.
os.environ.setdefault("OPENBTK_LOG_LEVEL", "warning")

'warning'

## Two synthetic patients

Bundles in the shape Synthea's per-patient export produces. Patient 1 has
pneumonia; patient 2 has a hypertension diagnosis. (A FHIR resource's `display`
text is free text, so patient 1's carries a fictitious phone number - which we
will see de-identified at the end.)

In [2]:
import json
import pathlib
import tempfile


def bundle(pid, code, display, when, lab=None):
    entries = [
        {
            "resource": {
                "resourceType": "Patient",
                "id": pid,
                "gender": "female",
                "birthDate": "1950-05-01",
            }
        },
        {
            "resource": {
                "resourceType": "Encounter",
                "id": "enc-1",
                "status": "finished",
                "class": {"code": "IMP"},
                "subject": {"reference": f"Patient/{pid}"},
                "period": {"start": f"{when}T08:00:00Z", "end": f"{when}T20:00:00Z"},
            }
        },
        {
            "resource": {
                "resourceType": "Condition",
                "id": "cond-1",
                "subject": {"reference": f"Patient/{pid}"},
                "encounter": {"reference": "Encounter/enc-1"},
                "code": {
                    "coding": [
                        {
                            "system": "http://snomed.info/sct",
                            "code": code,
                            "display": display,
                        }
                    ]
                },
                "onsetDateTime": f"{when}T08:10:00Z",
            }
        },
    ]
    if lab:
        entries.append(
            {
                "resource": {
                    "resourceType": "Observation",
                    "id": "obs-1",
                    "status": "final",
                    "subject": {"reference": f"Patient/{pid}"},
                    "encounter": {"reference": "Encounter/enc-1"},
                    "code": {
                        "coding": [
                            {
                                "system": "http://loinc.org",
                                "code": "6690-2",
                                "display": "WBC",
                            }
                        ]
                    },
                    "valueQuantity": {"value": lab, "unit": "10*3/uL"},
                    "effectiveDateTime": f"{when}T08:20:00Z",
                    "referenceRange": [
                        {"low": {"value": 4.5}, "high": {"value": 11.0}}
                    ],
                }
            }
        )
    return {"resourceType": "Bundle", "type": "collection", "entry": entries}


folder = pathlib.Path(tempfile.mkdtemp())  # scratch folder for the bundles
(folder / "p1.json").write_text(
    json.dumps(
        bundle(
            "p1",
            "385093006",
            "Community-acquired pneumonia (call (555) 010-2345)",
            "2024-03-14",
            lab=14.2,
        )
    )
)
(folder / "p2.json").write_text(
    json.dumps(bundle("p2", "38341003", "Hypertension", "2024-04-02"))
)
sorted(p.name for p in folder.iterdir())

['p1.json', 'p2.json']

## Load and normalise

`FHIRLoader` streams one `PatientRecord` per bundle. `TemporalNormalizer` anchors
events to their encounter.

In [3]:
from openbtk.data.ehr.fhir import FHIRLoader
from openbtk.data.ehr.temporal import TemporalNormalizer

patients = [TemporalNormalizer().process(p) for p in FHIRLoader().load(str(folder))]
for p in patients:
    print(
        p.patient_id,
        "|",
        len(p.encounters),
        "encounter(s),",
        len(p.conditions),
        "condition(s),",
        len(p.observations),
        "lab(s)",
    )

p1 | 1 encounter(s), 1 condition(s), 1 lab(s)
p2 | 1 encounter(s), 1 condition(s), 0 lab(s)


## Select a cohort

Predicates compose, and the builder streams - it never holds the whole source.

In [4]:
from openbtk.data.ehr.cohort import CohortBuilder, has_condition

pneumonia = list(CohortBuilder(patients).include(has_condition("385093006")))
not_pneumonia = list(CohortBuilder(patients).exclude(has_condition("385093006")))
print("pneumonia:", [p.patient_id for p in pneumonia])
print("everyone else:", [p.patient_id for p in not_pneumonia])

pneumonia: ['p1']
everyone else: ['p2']


## A patient as text

`PatientTimelineSerializer` renders a record as a normal `ClinicalTextRecord`.
That is the seam between the EHR and clinical-text modalities.

In [5]:
from openbtk.pipelines import PatientTimelineSerializer

timeline = PatientTimelineSerializer().serialize(pneumonia[0])
print(timeline.text)

2024-03-14 | Encounter  | Imp encounter
2024-03-14 | Condition  | Community-acquired pneumonia (call (555) 010-2345) (SNOMED 385093006)
2024-03-14 | Lab        | WBC 14.2 10*3/uL (ref 4.5-11.0) [HIGH] (LOINC 6690-2)


## De-identify and chunk it, like any note

The phone number embedded in the diagnosis text is caught by the same
de-identification step, because the timeline is just text.

In [6]:
from openbtk.data.clinical_text.chunking import FixedTokenChunker
from openbtk.data.clinical_text.preprocessing import DeidPreprocessor
from openbtk.deid.schemas import DeidMode

clean = DeidPreprocessor(mode=DeidMode.REDACT).process(timeline)
assert "010-2345" not in clean.text
chunks = list(FixedTokenChunker(max_tokens=30).chunk(clean))
print(len(chunks), "chunk(s); first:")
print(chunks[0].text)

1 chunk(s); first:
2024-03-14 | Encounter  | Imp encounter
2024-03-14 | Condition  | Community-acquired pneumonia (call [REDACTED]) (SNOMED 385093006)
2024-03-14 | Lab        | WBC 14.2 10*3/uL (ref 4.5-11.0) [HIGH] (LOINC 6690-2)


## Limits

The loader is exercised on Synthea-*shaped* bundles and against `fhir.resources`
validation - not a full real Synthea export or a hospital FHIR server. See the
[EHR guide](https://openbtk.org/openbtk-core/dev/guides/ehr/) for `OMOPLoader`
(Parquet only, source-value codes) and the EHR guardrails.